<a href="https://colab.research.google.com/github/leninathikam/AI-engineering-Foundations-to-Agents/blob/main/llm_playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Project 1: Build an LLM Playground

Welcome! In this project, you’ll learn foundations of large language models (LLMs). We’ll keep the code minimal and the explanations high‑level so that anyone who can run a Python cell can follow along.  

We'll be using Google Colab for this project. Colab is a free, browser-based platform that lets you run Python code and machine learning models without installing anything on your local computer. Click the button below to open this notebook directly in Google Colab and get started!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bytebyteai/ai-eng-projects/blob/main/project_1/lm_playground.ipynb)

---
## Learning Objectives  
* **Tokenization** and how raw text is tokenized into a sequene of discrete tokens
* Inspect **GPT2** and **Transformer architecture**
* Loading pre-trained LLMs using **Hugging Face**
* **Decoding strategies** to generate text from LLMs
* Completion versus **intrusction fine-tuned** LLMs


Let's get started!

In [1]:
import torch, transformers, tiktoken
print("torch", torch.__version__, "| transformers", transformers.__version__)

torch 2.10.0+cu128 | transformers 5.0.0


# 1 - Tokenization

A neural network can’t digest raw text. They need **numbers**. Tokenization is the process of converting text into IDs. In this section, you'll learn how tokenization is implemented in practice.

Tokenization methods generally fall into three categories:
1. Word-level
2. Character-level
3. Subword-level

### 1.1 - Word‑level tokenization

Split text on whitespace and store each **word** as a token.

In [2]:
# 1. Tiny corpus
corpus = [
    "The quick brown fox jumps over the lazy dog",
    "Tokenization converts text to numbers",
    "Large language models predict the next token"
]

# 2. Build the vocabulary
PAD, UNK = "[PAD]", "[UNK]"
vocab = []
word2id = {}
id2word = {}

"""
YOUR CODE HERE
"""
words = set()
for doc in corpus:
  words.update(doc.lower().split())
vocab = [PAD, UNK] + sorted(words)
word2id = {word: i for i, word in enumerate(vocab)}
id2word = {i: word for word, i in word2id.items()}


print(f"Vocabulary size: {len(vocab)} words")
print("First 15 vocab entries:", vocab[:15])

# 3. Encode / decode
def encode(text):
    """
    YOUR CODE HERE
    """
    return [word2id.get(word, word2id[UNK]) for word in text.lower().split()]

def decode(ids):
    """
    YOUR CODE HERE
    """
    return " ".join(id2word[i] for i in ids if i != word2id[PAD])

# 4. Demo
sample = "The brown unicorn jumps"
ids = encode(sample)
recovered = decode(ids)

print("\nInput text :", sample)
print("Token IDs  :", ids)
print("Decoded    :", recovered)

Vocabulary size: 21 words
First 15 vocab entries: ['[PAD]', '[UNK]', 'brown', 'converts', 'dog', 'fox', 'jumps', 'language', 'large', 'lazy', 'models', 'next', 'numbers', 'over', 'predict']

Input text : The brown unicorn jumps
Token IDs  : [17, 2, 1, 6]
Decoded    : the brown [UNK] jumps


Word-level tokenization has two major limitations:
1. Large vocabulary size
2. Out-of-vocabulary (OOV) issue

### 1.2 - Character‑level tokenization

Every single character (including spaces and emojis) gets its own ID. This guarantees zero out‑of‑vocabulary issues but very long sequences.

In [3]:
# 1. Build a fixed vocabulary
import string

letters = list(string.ascii_lowercase + string.ascii_uppercase)  # a–z + A–Z
special = ["[PAD]", "[UNK]"]  # padding + unknown
vocab = special + letters

char2id = {ch: idx for idx, ch in enumerate(vocab)}
id2char = {idx: ch for ch, idx in char2id.items()}

print(f"Vocabulary size: {len(vocab)} (52 letters + 2 specials)")

# 2. Encode / decode
def encode(text):
    """Convert text → list of IDs (unknown chars → [UNK])."""
    unk_id = char2id["[UNK]"]
    return [char2id.get(ch, unk_id) for ch in text]

def decode(ids):
    """Convert list of IDs."""
    return "".join(id2char[i] for i in ids if i != char2id["[PAD]"])

# 3. Demo
sample = "Hello"
ids = encode(sample)
recovered = decode(ids)

print("\nInput text :", sample)
print("Token IDs  :", ids)
print("Decoded    :", recovered)


Vocabulary size: 54 (52 letters + 2 specials)

Input text : Hello
Token IDs  : [35, 6, 13, 13, 16]
Decoded    : Hello


### 1.3 - Subword‑level tokenization

Sub-word methods such as `Byte-Pair Encoding (BPE)`, `WordPiece`, and `SentencePiece` **learn** the most common character and gorup them into new tokens. For example, the word `unbelievable` might turn into three tokens: `["un", "believ", "able"]`. This approach strikes a balance between word-level and character-level methods and fix their limitations.

For example, `BPE` algorithm forms the vocabulary using the following steps:
1. **Start with bytes** → every character is its own token.  
2. **Count all adjacent pairs** in a huge corpus.  
3. **Merge the most frequent pair** into a new token.  
   *Repeat steps 2-3* until you hit the target vocab size (e.g., 50 k).

Let's see `BPE` in practice.

In [4]:
# 1. Load a pretrained BPE tokenizer (GPT-2 uses BPE).
# Refer to  https://huggingface.co/docs/transformers/en/fast_tokenizers

from transformers import AutoTokenizer

bpe_tok = AutoTokenizer.from_pretrained("gpt2")

print("Vocab size:", bpe_tok.vocab_size)
print("Special tokens:", bpe_tok.all_special_tokens)

# 2. Encode / decode
def encode(text):
    return bpe_tok.encode(text)

def decode(ids):
    return bpe_tok.decode(ids)

# 3. Demo
sample = "Unbelievable tokenization powers! 🚀"
ids = encode(sample)
recovered = decode(ids)

print("\nInput text :", sample)
print("Token IDs  :", ids)
print("Tokens     :", bpe_tok.convert_ids_to_tokens(ids))
print("Decoded    :", recovered)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size: 50257
Special tokens: ['<|endoftext|>']

Input text : Unbelievable tokenization powers! 🚀
Token IDs  : [3118, 6667, 11203, 540, 11241, 1634, 5635, 0, 12520, 248, 222]
Tokens     : ['Un', 'bel', 'iev', 'able', 'Ġtoken', 'ization', 'Ġpowers', '!', 'ĠðŁ', 'ļ', 'Ģ']
Decoded    : Unbelievable tokenization powers! 🚀


### 1.4 - TikToken

`tiktoken` is a production-ready library which offers high‑speed tokenization used by OpenAI models.  
Let's compare the older **gpt2** encoding with the newer **cl100k_base** used in GPT‑4.

In [ ]:
# Use gpt2 and cl100k_base to encode and decode the following text
# Refer to https://github.com/openai/tiktoken
import tiktoken

sentence = "The 🌟 star-player scored 40 points!"

"""
YOUR CODE HERE
"""

encodings = [
    ("gpt2", tiktoken.get_encoding("gpt2")),
    ("cl100k_base", tiktoken.get_encoding("cl100k_base")),
]

for name, enc in encodings:
    print(f"\n=== {name} ===")
    print("Vocabulary size:", enc.n_vocab)

    # Encode the sample sentence
    ids = enc.encode(sentence)
    tokens = [enc.decode([i]) for i in ids]
    print(f"Sentence splits into {len(ids)} tokens:")
    print(list(zip(tokens, ids)))

    # Show a few arbitrary token→ID examples from the vocab
    some_ids = [0, 1, 2, 198, 50256]
    print("Sample tokens from the vocabulary:")
    print([(enc.decode([i]), i) for i in some_ids])

Experiment: try new sentences, emojis, code snippets, or other languages. If you are interested, try implementing the BPE algorithm yourself.

### 1.5 - Key Takeaways

* **Word‑level**: simple but brittle (OOV problems).  
* **Character‑level**: robust but produces long sequences.  
* **BPE / Byte‑Level BPE**: middle ground used by most LLMs.  
* **tiktoken**: shows how production models tokenize with pre‑trained sub‑word vocabularies.

# 2. What is a Language Model?

At its core, a **language model (LM)** is just a *very large* mathematical function built from many neural-network layers.  
Given a sequence of tokens `[t₁, t₂, …, tₙ]`, it learns to output a probability for the next token `tₙ₊₁`.


Each layer applies a simple operation (matrix multiplication, attention, etc.). Stacking hundreds of these layers lets the model capture patterns and statistical relations from text. The final output is a vector of scores that says, “how likely is each possible token to come next?”

> Think of the whole network as **one gigantic equation** whose parameters were tuned during training to minimize prediction error.



### 2.1 - A Single `Linear` Layer

Before we explore Transformer, let’s start tiny:

* A **Linear layer** performs `y = Wx + b`  
  * `x` – input vector  
  * `W` – weight matrix (learned)  
  * `b` – bias vector (learned)

Although this looks basic, chaining thousands of such linear transforms (with nonlinearities in between) gives neural nets their expressive power.


In [ ]:
import torch.nn as nn
class Linear(nn.Module):
    def __init__(self, in_features, out_features):
        super(Linear, self).__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.randn(out_features))

    def forward(self, x):
        return torch.matmul(x, self.weight.t()) + self.bias

lin = Linear(3, 2)
x = torch.tensor([1.0, -1.0, 0.5])
print("Input :", x)
print("Weights:", lin.weight)
print("Bias   :", lin.bias)
print("Output :", lin(x))

In [ ]:
import torch.nn as nn, torch

lin = nn.Linear(3, 2)
x = torch.tensor([1.0, -1.0, 0.5])
print("Input :", x)
print("Weights:", lin.weight)
print("Bias   :", lin.bias)
print("Output :", lin(x))


### 2.2 - A `Transformer` Layer

Most LLMs are a **stack of identical Transformer blocks**. Each block fuses two main components:

| Step | What it does | Where it lives in code |
|------|--------------|------------------------|
| **Multi-Head Self-Attention** | Every token looks at every other token and decides *what matters*. | `block.attn` |
| **Feed-Forward Network (MLP)** | Re-mixes information token-by-token. | `block.mlp` |

Below, we load the smallest public GPT-2 (124 M parameters), grab its *first* block, and inspect the pieces.


In [ ]:
import torch
from transformers import GPT2LMHeadModel

# Load the 124 M-parameter GPT-2
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
block = gpt2.transformer.h[0] # GPT-2 has 12 such layers
for name, module in block.named_children():
    print(f"{name:7s} → {module.__class__.__name__}")

In [ ]:
print("=== First Transformer Block ===")
print(block, "\n")

In [ ]:
# Run a tiny forward pass through one block
seq_len = 8
dummy_tokens = torch.randint(0, gpt2.config.vocab_size, (1, seq_len))
with torch.no_grad():
    # Embed tokens + positions the same way GPT-2 does
    hidden = (
        gpt2.transformer.wte(dummy_tokens) +
        gpt2.transformer.wpe(torch.arange(seq_len))
    )
    # Forward through one layer
    out = block(hidden, layer_past=None, use_cache=False)[0]
print("\nOutput shape :", out.shape) # (batch, seq_len, hidden_size)

### 2.3 - Inside GPT-2

GPT-2 is just many of those modules arranged in a repeating *block*. Let's print the modules inside the Transformer.

In [ ]:
for name, module in gpt2.transformer.named_children():
    print(f"{name:7s} → {module.__class__.__name__}")

As you can see, the Transformer holds various modules, arranged from a list of blocks (`h`). The following table summarizes these modules:

| Step | What it does | Why it matters |
|------|--------------|----------------|
| **Token → Embedding** | Converts IDs to vectors | Gives the model a numeric “handle” on words |
| **Positional Encoding** | Adds “where am I?” info | Order matters in language |
| **Multi-Head Self-Attention** | Each token asks “which other tokens should I look at?” | Lets the model relate words across a sentence |
| **Feed-Forward Network** | Two stacked Linear layers with a non-linearity | Mixes information and adds depth |
| **LayerNorm & Residual** | Stabilize training and help gradients flow | Keeps very deep networks trainable |


### 2.4 LLM's output

Passing a token sequence through an **LLM** yields a tensor of **logits** with shape  
`(batch_size, seq_len, vocab_size)`.  
Applying `softmax` on the last dimension turns those logits into probabilities.

The cell below feeds an 8-token dummy sequence, prints the logits shape, and shows the five most likely next tokens for the final position.


In [ ]:
import torch, torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

try:
    gpt2
except NameError:
    gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

# Tokenize input text
text = "Hello my name"
input_ids = tokenizer(text, return_tensors="pt").input_ids  # shape: (1, seq_len)


with torch.no_grad():
    logits = gpt2(input_ids).logits          # (1, seq_len, vocab_size)

print("Logits shape :", logits.shape)

# Predict next token
probs = F.softmax(logits[0, -1], dim=-1)     # (vocab_size,)
topk = torch.topk(probs, 5)

print("\nTop-5 predictions for the next token:")
for idx, p in zip(topk.indices.tolist(), topk.values.tolist()):
    print(f"{tokenizer.decode([idx]):>10s}  —  {p:.4f}")


### 2.5 - Key Takeaway

A language model is nothing mystical: it’s a *huge composition* of small, understandable layers trained to predict the next token in a sequence of tokens.

# 3 - Generation
Once an LLM is trained to predict the probabilities, we can generate text from the model. This process is called decoding or sampling.

At each step, the LLM outputs a **probability distribution** over the next token. It is the job of the decoding algorithm to pick the next token, and move on to the next token. There are different decoding algorithms and hyper-parameters to control the generaiton:
* **Greedy** → pick the single highest‑probability token each step (safe but repetitive).  
* **Top‑k / Nucleus (top‑p)** → sample from a subset of likely tokens (adds variety).
* **beam** -> applies beam search to pick tokens
* **Temperature** → a *creativity* knob. Higher values flatten the probability distribution.

### 3.1 - Greedy decoding

In [ ]:
# These are "auto" classes from the Hugging Face transformers library. They
# automatically figure out the correct architecture for the model you want to
# load (e.g., GPT2Tokenizer, GPT2LMHeadModel) just from its name. "CausalLM"
# means the model is designed for predicting the next token in a sequence.
from transformers import AutoTokenizer, AutoModelForCausalLM
MODELS = {
    "gpt2": "gpt2",
}
tokenizers, models = {}, {}

# This is a crucial step for performance. It checks if PyTorch can access a CUDA-enabled GPU.
# If a GPU is found, device is set to "cuda".
# If not, it defaults to "cpu".
# Calculations will run much faster on a GPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

# This loop iterates through the MODELS dictionary. For your example, key will
# be "gpt2" and mid (model ID) will also be "gpt2"
# For each model, there are some preparations need to be done.
for key, mid in MODELS.items():
    # Downloads (if not cached) and loads the tokenizer associated with the
    # model ID. The tokenizer's job is to convert text into numerical IDs.
    tok = AutoTokenizer.from_pretrained(mid)
    # Downloads and loads the actual pre-trained model weights.
    # .eval() is very important. It puts the model into evaluation mode. This
    # disables layers like dropout that are only used during training, ensuring
    # consistent and deterministic output.
    # .to(device) moves the entire model and all its parameters to the selected
    # device (either the GPU or CPU).
    mdl = AutoModelForCausalLM.from_pretrained(mid).eval().to(device)

    # This is a common and necessary fix for models like GPT-2. These models
    # were not trained with a specific padding token, which is needed when you
    # process multiple texts of different lengths in a single batch.
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token # This is a standard workaround.
    mdl.config.pad_token_id = tok.pad_token_id
    # This line stores the fully prepared tokenizer (tok) and model (mdl) into
    # their respective dictionaries, using the friendly name ("gpt2") as the key
    # for easy access later.
    tokenizers[key], models[key] = tok, mdl
    print(f"Loaded {mid} as {key}")

def generate(model_key, prompt, strategy="greedy", max_new_tokens=100):
    tok, mdl = tokenizers[model_key], models[model_key]
    # tok(prompt, return_tensors="pt") tokenizes your input prompt and returns
    # it as a PyTorch ("pt") tensor.
    # .to(mdl.device) moves the input tensor to the same device (GPU or CPU)
    # where the model is stored. This is essential to prevent errors.
    # enc is the output from the tokenizer. It's a dictionary-like object that
    # looks something like this: {'input_ids': tensor([[...]]), 'attention_mask': tensor([[...]])}
    enc = tok(prompt, return_tensors="pt").to(mdl.device)

    # ** operator is a powerful Python feature called dictionary unpacking. It
    # essentially "unpacks" the key-value pairs from a dictionary and passes
    # them as keyword arguments.
    # Python expands **enc into input_ids=tensor([[...]]), attention_mask=tensor([[...]]).
    # The line effectively becomes:
    # gen_args = dict(
    #   input_ids=tensor([[...]]),          # From **enc
    #   attention_mask=tensor([[...]]),     # From **enc
    #   max_new_tokens=100,                 # Added argument
    #   pad_token_id=50256                  # Added argument
    # )
    # It's a concise way to create a new dictionary by merging the contents of
    # an existing dictionary (enc) with a few new key-value pairs.
    # The single asterisk (*) works on lists and tuples.
    # * (Positional Unpacking): Unpacks a list into positional arguments.
    # my_list = [10, 20, 30]
    # print(*my_list) # Same as print(10, 20, 30)
    # ** (Keyword Unpacking): Unpacks a dictionary into keyword arguments.
    # my_dict = {'a': 1, 'b': 2}
    # func(**my_dict) is the same as func(a=1, b=2)
    gen_args = dict(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id)
    # "greedy" is the most straightforward and deterministic strategy. At each
    # step, it simply picks the single word with the highest probability.
    # do_sample=False enables this.
    if strategy=="greedy":
        gen_args["do_sample"]=False
    # "top_k" introduces randomness. The model considers the top 50 most likely
    # words (top_k=50) and then randomly samples from that pool. This prevents
    # the model from picking very strange, low-probability words.
    # temperature=0.9 controls the randomness. A value less than 1 (like 0.9)
    # sharpens the probability distribution, making the model more confident and
    # less random. A value greater than 1 would make it more creative and random.
    elif strategy=="top_k":
        gen_args.update(dict(do_sample=True, top_k=50, temperature=0.9))
    # "top_p" (Nucleus Sampling) is often more effective. It chooses the
    # smallest set of words whose cumulative probability is at least 90%
    # (top_p=0.9) and then samples from that set. The size of this set can
    # change at each step, making it more adaptive than top_k.
    elif strategy=="top_p":
        gen_args.update(dict(do_sample=True, top_p=0.9, temperature=0.9))

    # mdl.generate(**gen_args) is the main event. It calls the model's generate
    # method, unpacking all the arguments we just configured. The out variable
    # will contain the tensor of token IDs for the complete sequence
    # (prompt + new text).
    out = mdl.generate(**gen_args)
    # tok.decode(...) is the final step. It takes the output tensor and converts
    # the sequence of numbers back into a human-readable string.
    # out[0] select the first (and only) result from the output batch.
    # skip_special_tokens=True removes special tokens like <|endoftext|> from
    # the final text, making it clean.
    return tok.decode(out[0], skip_special_tokens=True)



In [ ]:
tests=["Once upon a time","What is 2+2?", "Suggest a party theme."]
for prompt in tests:
    print(f"\n== GPT-2 | Greedy ==")
    print(generate("gpt2", prompt, "greedy", 80))



Naively picking the single best token every time has the following issues in practice:

* **Loop**: “The cat is is is…”  
* **Miss long-term payoff**: the highest-probability word *now* might paint you into a boring corner later.

### 3.2 - Top-k or top-p sampling

In [ ]:

tests=["Once upon a time","What is 2+2?", "Suggest a party theme."]
for prompt in tests:
    print(f"\n== GPT-2 | Top-p ==")
    print(generate("gpt2", prompt, "top-p", 40))


### 3.3 - Try It Yourself

1. Scroll to the list called `tests`.
2. Swap in your own prompts or tweak the decoding strategy.  
3. Re‑run the cell and compare the vibes.

> **Tip:** Try the same prompt with `greedy` vs. `top_p` (0.9) and see how the tone changes. Notice especially how small temperature tweaks can soften or sharpen the prose.

* `strategy`: `"greedy"`, `"beam"`, `"top_k"`, `"top_p"`  
* `temperature`: `0.2 – 2.0`  
* `k` or `p` thresholds



# 4 - Completion vs. Instruction-tuned LLMs

We have seen that we can use GPT2 model to pass an input text and generate a new text. However, this model only continues the provided text. It is not engaging in a dialouge-like conversation and cannot be helpful by answering instructions. On the other hand, **instruction-tuned LLMs** like `Qwen-Chat` go through an extra training stage called **post-training** after the base “completion” model is finished. Because of post-training step, an instruction-tuned LLM will:

* **Read the entire prompt as a request,** not just as text to mimic.  
* **Stay in dialogue mode**. Answer questions, follow steps, ask clarifying queries.  
* **Refuse or safe-complete** when instructions are unsafe or disallowed.  
* **Adopt a consistent persona** (e.g., “Assistant”) rather than drifting into story continuation.


### 4.1 - Qwen1.5-8B vs. GPT2

In the code below we’ll feed the same prompt to:

* **GPT-2 (completion-only)** – it will simply keep writing in the same style.  
* **Qwen-Chat (instruction-tuned)** – it will obey the instruction and respond directly.

Comparing the two outputs makes the difference easy to see.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
MODELS = {
    "gpt2": "gpt2",
    "qwen": "Qwen/Qwen1.5-1.8B-Chat",
    # "gemma": "google/gemma-3-270m"
}
tokenizers, models = {}, {}
device = "cuda" if torch.cuda.is_available() else "cpu"
for key, mid in MODELS.items():
    tok = AutoTokenizer.from_pretrained(mid)
    mdl = AutoModelForCausalLM.from_pretrained(mid).eval().to(device)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl.config.pad_token_id = tok.pad_token_id
    tokenizers[key], models[key] = tok, mdl
    print(f"Loaded {mid} as {key}")



We downloaded two tiny checkpoints: `GPT‑2` (124 M parameters) and `Qwen‑1.5‑Chat` (1.8 B). If the cell took a while, that was mostly network time. Models are stored locally after the first run.

Let's now generate text and compare two models.


In [ ]:

tests=[("Once upon a time","greedy"),("What is 2+2?","top_k"),("Suggest a party theme.","top_p")]
for prompt,strategy in tests:
    for key in ["gpt2","qwen"]:
        print(f"\n== {key.upper()} | {strategy} ==")
        print(generate(key,prompt,strategy,80))


# 5. (Optional) A Small LLM Playground

### 5.1 ‑ Interactive Playground

Enter a prompt, pick a model and decoding strategy, adjust the temperature, and press **Generate** to watch the model respond.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

# Make sure models and tokenizers are loaded
try:
    tokenizers
    models
except NameError:
    raise RuntimeError("Please run the earlier setup cells that load the models before using the playground.")

def generate_playground(model_key, prompt, strategy="greedy", temperature=1.0, max_new_tokens=100):
    tok, mdl = tokenizers[model_key], models[model_key]
    enc = tok(prompt, return_tensors="pt").to(mdl.device)
    gen_args = dict(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id)
    if strategy == "greedy":
        gen_args["do_sample"] = False
    elif strategy == "top_k":
        gen_args.update(dict(do_sample=True, top_k=50, temperature=temperature))
    elif strategy == "top_p":
        gen_args.update(dict(do_sample=True, top_p=0.9, temperature=temperature))
    else:
        raise ValueError("Unknown strategy")
    out = mdl.generate(**gen_args)
    return tok.decode(out[0], skip_special_tokens=True)

prompt_box = widgets.Textarea(
    value="Tell me a fun fact about space.",
    placeholder="Type your prompt here",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="120px")
)

model_dropdown = widgets.Dropdown(
    options=[("GPT‑2", "gpt2"), ("Qwen‑1.5‑Chat", "qwen")],
    value="gpt2",
    description="Model:"
)

strategy_dropdown = widgets.Dropdown(
    options=[("Greedy", "greedy"), ("Top‑k", "top_k"), ("Top‑p", "top_p")],
    value="greedy",
    description="Strategy:"
)

temperature_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=2.0,
    step=0.1,
    description="Temp:"
)

generate_button = widgets.Button(description="Generate", button_style="primary")
output_area = widgets.Output()

def on_generate(_):
    output_area.clear_output()
    with output_area:
        try:
            result = generate_playground(
                model_dropdown.value,
                prompt_box.value,
                strategy_dropdown.value,
                temperature_slider.value
            )
            display(Markdown(f"**Output:**\n\n{result}"))
        except Exception as e:
            print("Error:", e)

generate_button.on_click(on_generate)

ui = widgets.VBox([
    prompt_box,
    widgets.HBox([model_dropdown, strategy_dropdown, temperature_slider]),
    generate_button,
    output_area
])

display(ui)


## 🎉 Congratulations!

You’ve just learned, explored, and inspected a real **LLM**. In one project you:
* Learned how **tokenization** works in practice
* Used `tiktoken` library to load and experiment with most advanced tokenizers.
* Explored LLM architecture and inspected GPT2 blocks and layers
* Learned decoding strategies and used `top-p` to generate text from GPT2
* Loaded a powerful chat model, `Qwen1.5-8B` and generated text
* Built an LLM playground


👏 **Great job!** Take a moment to celebrate. You now have a working mental model of how LLMs work. The skills you used here power most LLMs you see everywhere.
